### NB_OTT_ExperimentsCluster

#### Objetivo:
##### Evaluar distintas configuraciones de DBSCAN utilizando el ground truth disponible en el dataset sintético.

##### Este notebook se utiliza durante la fase experimental y NO forma parte del pipeline operacional.

#### Resultados:
##### Los resultados de este experimento para obtener valores como el eps y min_samples se guardan en Gold.ClusteringEvaluation
##### Los parámetros seleccionados aquí se utilizan posteriormente en nuestro pipeline operacional para crear el cluster, en el notebook NB_OTT_TicketClustering.

In [2]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score
)

from pyspark.sql import functions as F

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 4, Finished, Available, Finished, False)

In [3]:
# ============================================================
# 2. Carga de tickets y embeddings
#
# A diferencia del notebook operacional, aquí cargamos
# incident_id e is_isolated porque son necesarios para
# evaluar los resultados contra el ground truth.
# ============================================================

df = (
    spark.table("Silver.Tickets")
    .select(
        "ticket_id",
        "incident_id",
        "is_isolated"
    )
    .join(
        spark.table("Silver.TicketEmbeddings")
        .select(
            "ticket_id",
            "embedding"
        ),
        on="ticket_id",
        how="inner"
    )
)

print("Tickets available for evaluation:", df.count())

display(
    df.limit(5)
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 5, Finished, Available, Finished, False)

Tickets available for evaluation: 1000


SynapseWidget(Synapse.DataFrame, a8118a1b-95ac-4438-ad49-e0e48bc8118d)

In [4]:
# ============================================================
# 3. Conversión a Pandas / NumPy
# ============================================================

pdf = df.toPandas()

X = np.vstack(
    pdf["embedding"]
    .apply(np.array)
    .values
)

print("Embedding matrix:", X.shape)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 6, Finished, Available, Finished, False)

Embedding matrix: (1000, 384)


In [5]:
# ============================================================
# 4. Construcción del ground truth
#
# Los tickets pertenecientes a un incidente usan incident_id.
#
# Los tickets aislados reciben un identificador único para
# evaluación, ya que conceptualmente cada uno representa un
# problema independiente.
#
# Esta variable se usa ÚNICAMENTE para evaluación.
# ============================================================

def build_ground_truth(row):
    if (
        bool(row["is_isolated"]) or
        pd.isna(row["incident_id"])
    ):
        return f"isolated_{row['ticket_id']}"

    return str(row["incident_id"])


pdf["evaluation_incident_id"] = (
    pdf.apply(
        build_ground_truth,
        axis=1
    )
)

y_true = (
    pdf["evaluation_incident_id"]
    .astype(str)
    .to_numpy()
)

print("Tickets:", len(y_true))
print(
    "Ground-truth groups:",
    pdf["evaluation_incident_id"].nunique()
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 7, Finished, Available, Finished, False)

Tickets: 1000
Ground-truth groups: 150


In [6]:
# ============================================================
# 5. Pairwise metrics
#
# Para cada pareja de tickets se comprueba:
#
# Ground truth:
#   ¿pertenecen al mismo incidente?
#
# Predicción:
#   ¿DBSCAN los ha colocado en el mismo cluster?
#
# Los tickets DBSCAN noise (-1) NO se consideran agrupados
# entre sí.
# ============================================================

def pairwise_metrics(
    y_true,
    y_pred
):
    tp = 0
    fp = 0
    fn = 0

    n = len(y_true)

    for i in range(n):

        for j in range(i + 1, n):

            same_true = (
                y_true[i] == y_true[j]
            )

            same_pred = (
                y_pred[i] != -1
                and
                y_pred[j] != -1
                and
                y_pred[i] == y_pred[j]
            )

            if same_true and same_pred:
                tp += 1

            elif (
                not same_true
                and same_pred
            ):
                fp += 1

            elif (
                same_true
                and not same_pred
            ):
                fn += 1

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return (
        precision,
        recall,
        f1
    )

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 8, Finished, Available, Finished, False)

In [7]:
# ============================================================
# 6. Espacio experimental
#
# Se evalúan distintas combinaciones de:
#
# eps:
#   controla la distancia máxima entre tickets vecinos.
#
# min_samples:
#   controla el número mínimo de tickets necesario para
#   formar una región densa.
# ============================================================

eps_values = np.arange(
    0.10,
    0.41,
    0.025
)

min_samples_values = [
    2,
    3,
    4,
    5
]

print("eps values:")
print(eps_values)

print(
    "min_samples:",
    min_samples_values
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 9, Finished, Available, Finished, False)

eps values:
[0.1   0.125 0.15  0.175 0.2   0.225 0.25  0.275 0.3   0.325 0.35  0.375
 0.4  ]
min_samples: [2, 3, 4, 5]


In [8]:
# ============================================================
# 7. Grid search
#
# Este es el bloque computacionalmente costoso que hemos
# eliminado del pipeline operacional.
# ============================================================

results = []

for current_eps in eps_values:

    for current_min_samples in min_samples_values:

        labels = DBSCAN(
            eps=float(current_eps),
            min_samples=int(
                current_min_samples
            ),
            metric="cosine"
        ).fit_predict(X)

        # ------------------------------------
        # Estadísticas básicas del clustering
        # ------------------------------------

        n_clusters = len(
            set(labels) - {-1}
        )

        n_noise = int(
            np.sum(labels == -1)
        )

        # ------------------------------------
        # Métricas globales
        # ------------------------------------

        ari = adjusted_rand_score(
            y_true,
            labels
        )

        nmi = normalized_mutual_info_score(
            y_true,
            labels
        )

        homogeneity = homogeneity_score(
            y_true,
            labels
        )

        completeness = completeness_score(
            y_true,
            labels
        )

        v_measure = v_measure_score(
            y_true,
            labels
        )

        # ------------------------------------
        # Métricas pairwise
        # ------------------------------------

        (
            pairwise_precision,
            pairwise_recall,
            pairwise_f1
        ) = pairwise_metrics(
            y_true,
            labels
        )

        # ------------------------------------
        # Registro del experimento
        # ------------------------------------

        results.append({
            "eps":
                float(current_eps),

            "min_samples":
                int(current_min_samples),

            "clusters":
                int(n_clusters),

            "noise":
                int(n_noise),

            "ARI":
                float(ari),

            "NMI":
                float(nmi),

            "homogeneity":
                float(homogeneity),

            "completeness":
                float(completeness),

            "v_measure":
                float(v_measure),

            "pairwise_precision":
                float(pairwise_precision),

            "pairwise_recall":
                float(pairwise_recall),

            "pairwise_f1":
                float(pairwise_f1)
        })


results_pdf = pd.DataFrame(
    results
)

print(
    "Configurations evaluated:",
    len(results_pdf)
)

display(
    results_pdf.head(10)
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 10, Finished, Available, Finished, False)

2026-09-14 18:06:34 [INFO] setting mlflow tracking uri to sds://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow, change this environment variable if you want log to other places
🏃 View run dreamy_atemoya_yh83gzzp at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/29dd04fe-05d8-4a47-809e-11b7210ca8df
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


🏃 View run icy_basket_k5q9wfhp at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/3be4680e-a3d2-4d20-900b-7027d919f4b1
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


🏃 View run sincere_bridge_fd18wdnw at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/d3d774de-04b3-4036-986e-c5e9499272ac
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


🏃 View run silver_kettle_pt6kjr0h at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/679528cc-07ef-4893-9bcc-14ffafe84437
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


...

🏃 View run affable_dream_3lq6205j at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/7db91c61-e324-4815-beff-1f1f1d7372b9
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


🏃 View run epic_office_d328bmwn at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/c08251d6-6112-44f0-a7e7-92977406d6c0
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


🏃 View run helpful_cushion_t4w0s189 at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde/runs/27920306-a873-47cc-93cc-f2cce93f13b0
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/64f79acc-c907-40c2-a029-8b24d23bec28/mlflow/#/experiments/d61f6089-9e57-4718-a115-a9caed5b8cde


Configurations evaluated: 52


SynapseWidget(Synapse.DataFrame, 860bb0f0-a844-4d7b-826f-1579b77b3c8b)

In [9]:
# ============================================================
# 8. Mejores configuraciones por métrica
# ============================================================

print("Best configurations by Pairwise F1")

display(
    results_pdf
    .sort_values(
        "pairwise_f1",
        ascending=False
    )
    .head(15)
)

print("Best configurations by ARI")

display(
    results_pdf
    .sort_values(
        "ARI",
        ascending=False
    )
    .head(15)
)

print("Best configurations by NMI")

display(
    results_pdf
    .sort_values(
        "NMI",
        ascending=False
    )
    .head(15)
)

display(
    results_pdf[
        [
            "eps",
            "min_samples",
            "clusters",
            "noise",
            "pairwise_precision",
            "pairwise_recall",
            "pairwise_f1"
        ]
    ]
    .sort_values(
        "pairwise_f1",
        ascending=False
    )
    .head(20)
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 11, Finished, Available, Finished, False)

Best configurations by Pairwise F1


SynapseWidget(Synapse.DataFrame, 5884ab59-73d3-4d72-8170-973f36e63789)

Best configurations by ARI


SynapseWidget(Synapse.DataFrame, 503a80fd-6b02-466c-9107-3476fe5d1145)

Best configurations by NMI


SynapseWidget(Synapse.DataFrame, 6c41b234-c0e6-43d9-aac8-8d94a3ec6be2)

SynapseWidget(Synapse.DataFrame, 91a2da19-649d-4cb1-ada9-1b6d6ef49291)

In [10]:
# ============================================================
# 9. Persistencia de Gold.ClusteringEvaluation
#
# Se conservan todas las configuraciones experimentales para:
# - análisis posterior
# - Power BI
# - trazabilidad
# - justificación de hiperparámetros
# ============================================================

results_pdf["algorithm"] = "DBSCAN"

results_pdf[
    "embedding_model"
] = "all-MiniLM-L6-v2"


results_spark = (
    spark.createDataFrame(
        results_pdf
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS Gold"
)

(
    results_spark.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "Gold.ClusteringEvaluation"
    )
)

print(
    "Experiments saved:",
    spark.table(
        "Gold.ClusteringEvaluation"
    ).count()
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 12, Finished, Available, Finished, False)

Experiments saved: 52


In [11]:
# ============================================================
# 10. Configuración seleccionada
#
# La configuración final no se considera un óptimo absoluto.
# Se selecciona como baseline de compromiso entre:
#
# - recuperación de tickets relacionados
# - fragmentación
# - sobreagrupamiento
# - número de grupos resultantes
#
# Configuración utilizada posteriormente por el pipeline:
#
#     eps = 0.25
#     min_samples = 3
# ============================================================

SELECTED_EPS = 0.25
SELECTED_MIN_SAMPLES = 3

selected_result = (
    results_pdf[
        (
            np.isclose(
                results_pdf["eps"],
                SELECTED_EPS
            )
        )
        &
        (
            results_pdf[
                "min_samples"
            ]
            ==
            SELECTED_MIN_SAMPLES
        )
    ]
)

display(selected_result)

print(
    "Selected baseline:"
)

print(
    f"eps={SELECTED_EPS}, "
    f"min_samples={SELECTED_MIN_SAMPLES}"
)

StatementMeta(, 74aededf-1d67-4f10-aa13-e0998920a54b, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d12848ed-4afe-4364-8f45-5a4e4ecce4d4)

Selected baseline:
eps=0.25, min_samples=3
